In [0]:
%pip install kaggle
dbutils.library.restartPython()   # reinicia Python para que el paquete quede disponible

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
CATALOG, SCHEMA = "workspace", "spotify"   # "workspace" es el catálogo por defecto en Free Edition
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

In [0]:
import os
dbutils.widgets.text("kaggle_user", "")
dbutils.widgets.text("kaggle_key", "")
os.environ["KAGGLE_USERNAME"] = dbutils.widgets.get("kaggle_user")
os.environ["KAGGLE_KEY"]      = dbutils.widgets.get("kaggle_key")

## importar despues de definir
already_downloaded = any(f.name == "data/" for f in dbutils.fs.ls(RAW_PATH))
if already_downloaded:
    print("Los datos ya están en el volumen: se omite la descarga")
else:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.dataset_download_files("amith1707/spotify-music-features-dataset",
                               path=RAW_PATH, unzip=True)
display(dbutils.fs.ls(RAW_PATH))

Los datos ya están en el volumen: se omite la descarga


path,name,size,modificationTime
dbfs:/Volumes/workspace/spotify/raw/LICENSE.txt,LICENSE.txt,25817,1794332414000
dbfs:/Volumes/workspace/spotify/raw/README.md,README.md,19081,1794332414000
dbfs:/Volumes/workspace/spotify/raw/data/,data/,0,1794525063841
dbfs:/Volumes/workspace/spotify/raw/metadata/,metadata/,0,1794525063841
dbfs:/Volumes/workspace/spotify/raw/notebook/,notebook/,0,1794525063841


In [0]:
# 1. Archivos de cada subcarpeta de data/, con su tamaño
for sub in ["full", "features", "splits", "subsets"]:
    print(f"--- {sub} ---")
    for f in dbutils.fs.ls(f"{RAW_PATH}/data/{sub}"):
        print(f"{f.name:55s} {f.size/1024/1024:9.2f} MB")

# 2. El resto del README (los volúmenes también se leen como archivos normales de Python)
with open(f"{RAW_PATH}/README.md") as fh:
    readme = fh.read()
print(readme[6000:14000])

# 3. Los metadatos: quizá incluyen los tipos de cada columna
with open(f"{RAW_PATH}/metadata/metadata.json") as fh:
    print(fh.read()[:3000])

--- full ---
spotify_full.csv                                           779.13 MB
--- features ---
spotify_albums.csv                                           2.51 MB
spotify_artist_features.csv                                 11.46 MB
spotify_artists.csv                                          7.88 MB
spotify_audio_features.csv                                 257.88 MB
spotify_decade_trends.csv                                    0.00 MB
spotify_features.csv                                       435.05 MB
spotify_genre_tags.csv                                     136.23 MB
spotify_playlist_cooccurrence.csv                          182.70 MB
spotify_popularity_metrics.csv                             167.32 MB
taste_cluster_report.csv                                     0.02 MB
--- splits ---
spotify_test.csv                                            77.90 MB
spotify_train.csv                                          623.32 MB
spotify_val.csv                                           

In [0]:
from pyspark.sql import functions as F

def ingest_csv_to_bronze(path, table, multiline=False):
    """Lee un CSV tal cual (todo como texto) y lo guarda como tabla Delta en bronce."""
    df = (spark.read
          .option("header", True)
          .option("escape", '"')              # trata "" dentro de un campo como una comilla literal
          .option("multiLine", multiline)
          .csv(path)
          .withColumn("_ingest_ts", F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_path")))
    full_name = f"{CATALOG}.{SCHEMA}.{table}"
    (df.write.mode("overwrite")
       .option("overwriteSchema", True)       # permite volver a ejecutar aunque cambien las columnas
       .saveAsTable(full_name))
    n = spark.table(full_name).count()
    print(f"{full_name:45s} {n:>12,} filas")
    return n

In [0]:
EXPECTED = 1_182_354   # row_count según metadata.json
n = ingest_csv_to_bronze(f"{RAW_PATH}/data/full/spotify_full.csv", "bronze_tracks")
print("OK" if n == EXPECTED else f"⚠️ Diferencia de {n - EXPECTED:+,} filas")

workspace.spotify.bronze_tracks                  1,182,354 filas
OK


In [0]:
for f in dbutils.fs.ls(f"{RAW_PATH}/data/features"):
    table = "bronze_" + f.name.removesuffix(".csv").removeprefix("spotify_")
    ingest_csv_to_bronze(f.path, table)

workspace.spotify.bronze_albums                     18,355 filas
workspace.spotify.bronze_artist_features            69,414 filas
workspace.spotify.bronze_artists                    69,414 filas
workspace.spotify.bronze_audio_features          1,182,354 filas
workspace.spotify.bronze_decade_trends                   8 filas
workspace.spotify.bronze_features                1,182,354 filas
workspace.spotify.bronze_genre_tags              1,164,266 filas
workspace.spotify.bronze_playlist_cooccurrence    1,261,407 filas
workspace.spotify.bronze_popularity_metrics      1,182,354 filas
workspace.spotify.bronze_taste_cluster_report          246 filas


In [0]:
spark.sql(f"""
  SELECT COUNT(*) AS filas_con_energy_invalida
  FROM {CATALOG}.{SCHEMA}.bronze_tracks
  WHERE energy IS NOT NULL AND try_cast(energy AS DOUBLE) IS NULL
""").show()

+-------------------------+
|filas_con_energy_invalida|
+-------------------------+
|                        0|
+-------------------------+



In [0]:
for t in spark.catalog.listTables(f"{CATALOG}.{SCHEMA}"):
    if t.name.startswith("bronze_"):
        cols = [c for c in spark.table(f"{CATALOG}.{SCHEMA}.{t.name}").columns if not c.startswith("_")]
        print(f"\n{t.name} ({len(cols)} columnas):\n  {', '.join(cols)}")


bronze_albums (12 columnas):
  album_id, title, artist_id, album_type, release_year, release_date, track_count, label, upc, spotify_id, musicbrainz_id, source_dataset

bronze_artist_features (17 columnas):
  artist_id, track_artist, catalog_size, mean_energy, mean_valence, mean_danceability, mean_tempo, career_start_year, career_end_year, mean_popularity, hit_rate, genre_l1_primary, genre_breadth, longevity_score, artist_stream_mean_log, sonic_evolution_score, career_span_years

bronze_artists (15 columnas):
  artist_id, name, name_normalized, country_of_origin, career_start_year, primary_genre_l1, primary_genre_l2, spotify_followers, spotify_popularity, monthly_listeners, musicbrainz_id, lastfm_listeners, lastfm_play_count, is_solo_artist, source_dataset

bronze_audio_features (19 columnas):
  track_id, energy, valence, danceability, acousticness, instrumentalness, speechiness, liveness, loudness, tempo, key, key_name, mode, mode_name, time_signature, duration_ms, tempo_confidence, f

In [0]:
import json
with open(f"{RAW_PATH}/metadata/metadata.json") as fh:
    meta = json.load(fh)
expected_types = {c["name"]: c["type"] for c in meta["schema"]}
print("Tipos que usa el autor:", sorted(set(expected_types.values())))

Tipos que usa el autor: ['bool', 'float', 'int', 'string']


In [0]:
tracks = spark.table(f"{CATALOG}.{SCHEMA}.bronze_tracks")
data_cols = [c for c in tracks.columns if not c.startswith("_")]

def is_missing(c):
    return F.col(c).isNull() | F.lower(F.trim(F.col(c))).isin("", "nan", "null", "none")
nulls = (tracks.select([F.sum(is_missing(c).cast("int")).alias(c) for c in data_cols])
               .toPandas().T.reset_index())      # reset_index convierte el índice en una columna visible
nulls.columns = ["columna", "faltantes"]
nulls["pct"] = (nulls["faltantes"] / EXPECTED * 100).round(2)
display(nulls[nulls["faltantes"] > 0].sort_values("pct", ascending=False))


columna,faltantes,pct
taste_cluster,1180175,99.82
streams_source,1172754,99.19
chart_peak_position,1167441,98.74
weeks_on_chart,1167441,98.74
chart_date_first,1167441,98.74
anomaly_flags,1160967,98.19
release_date,1159764,98.09
album_id,1156961,97.85
album_name,1156965,97.85
title_normalized,25901,2.19


In [0]:
spark.sql(f"""
  SELECT COUNT(*)                                          AS filas,
         COUNT(DISTINCT track_id)                          AS track_id_unicos,
         COUNT(DISTINCT spotify_id)                        AS spotify_id_unicos,
         COUNT(DISTINCT title_normalized, track_artist)    AS titulo_artista_unicos
  FROM {CATALOG}.{SCHEMA}.bronze_tracks
""").show()

+-------+---------------+-----------------+---------------------+
|  filas|track_id_unicos|spotify_id_unicos|titulo_artista_unicos|
+-------+---------------+-----------------+---------------------+
|1182354|        1182354|          1182354|              1136625|
+-------+---------------+-----------------+---------------------+



In [0]:
# Celda 13: ¿qué valores no se pueden convertir a su tipo esperado?
# Requiere: celda 10 (expected_types) y celda 11 (tracks, data_cols, is_missing)

# 1. Traducir los tipos del autor a tipos de Spark ("string" no se revisa)
spark_type = {"int": "BIGINT", "float": "DOUBLE", "bool": "BOOLEAN"}
to_check = {c: spark_type[t] for c, t in expected_types.items()
            if t in spark_type and c in data_cols}
print(f"Columnas revisadas: {len(to_check)}")

# 2. Contar, por columna, los valores NO vacíos que fallan al convertirse (una sola pasada)
fails = (tracks.select([
            F.sum((~is_missing(c) & F.expr(f"try_cast(`{c}` AS {t})").isNull()).cast("int")).alias(c)
            for c, t in to_check.items()])
         .toPandas().T.reset_index())
fails.columns = ["columna", "no_convertibles"]
fails["tipo_esperado"] = fails["columna"].map(to_check)

# 3. Mostrar solo las columnas con problemas, o un mensaje si no hay ninguna
bad = fails[fails["no_convertibles"] > 0]
if bad.empty:
    print("✅ Todos los valores no vacíos se pueden convertir a su tipo esperado")
else:
    display(bad)

Columnas revisadas: 58
✅ Todos los valores no vacíos se pueden convertir a su tipo esperado


In [0]:
s = f"{CATALOG}.{SCHEMA}"
spark.sql(f"""
SELECT
  (SELECT COUNT(*) FROM {s}.bronze_artist_features)                              AS filas_artist_features,
  (SELECT COUNT(DISTINCT track_artist) FROM {s}.bronze_artist_features)          AS nombres_unicos,
  (SELECT COUNT(*) FROM {s}.bronze_tracks t
     LEFT ANTI JOIN {s}.bronze_albums a ON t.album_id = a.album_id)              AS tracks_sin_album,
  (SELECT COUNT(*) FROM {s}.bronze_tracks t
     LEFT ANTI JOIN {s}.bronze_artist_features af ON t.track_artist = af.track_artist) AS tracks_sin_artista,
  (SELECT COUNT(*) FROM {s}.bronze_tracks t
     LEFT ANTI JOIN {s}.bronze_features f ON t.track_id = f.track_id)            AS tracks_sin_features
""").show(vertical=True)

-RECORD 0------------------------
 filas_artist_features | 69414   
 nombres_unicos        | 69414   
 tracks_sin_album      | 1159528 
 tracks_sin_artista    | 19      
 tracks_sin_features   | 0       



In [0]:
s = f"{CATALOG}.{SCHEMA}"

# 1. Comparar visualmente el formato de los IDs de álbum en las dos tablas
print("album_id en tracks:")
spark.sql(f"SELECT album_id FROM {s}.bronze_tracks WHERE album_id IS NOT NULL LIMIT 5").show(truncate=False)
print("Claves en albums:")
spark.sql(f"SELECT album_id, spotify_id FROM {s}.bronze_albums LIMIT 5").show(truncate=False)

# 2. Medir: ¿cuántas canciones se unen con cada clave candidata?
spark.sql(f"""
SELECT
  (SELECT COUNT_IF(album_id IS NULL OR trim(album_id) = '') FROM {s}.bronze_tracks) AS album_id_vacio,
  (SELECT COUNT(*) FROM {s}.bronze_tracks t
     LEFT SEMI JOIN {s}.bronze_albums a ON t.album_id = a.album_id)                AS match_album_id,
  (SELECT COUNT(*) FROM {s}.bronze_tracks t
     LEFT SEMI JOIN {s}.bronze_albums a ON t.album_id = a.spotify_id)              AS match_spotify_id
""").show(vertical=True)

# 3. Las 19 canciones sin artista: ¿son registros casi vacíos?
spark.sql(f"""
SELECT t.track_id, t.title, t.track_artist, t.has_audio_features, t.energy, t.genre_l1
FROM {s}.bronze_tracks t
LEFT ANTI JOIN {s}.bronze_artist_features af ON t.track_artist = af.track_artist
""").show(truncate=False)

album_id en tracks:
+----------------------+
|album_id              |
+----------------------+
|7I0tjwFtxUwBC1vgyeMAax|
|2ZfHkwHuoAZrlz7RMj0PDz|
|4g1ZRSobMefqF6nelkgibi|
|3nR9B40hYLKLcR0Eph3Goc|
|7mKevNHhVnZER3BLgI8O4F|
+----------------------+

Claves en albums:
+----------------------+----------------------+
|album_id              |spotify_id            |
+----------------------+----------------------+
|000YOrgQoB5IiiH95Yb8vY|000YOrgQoB5IiiH95Yb8vY|
|000f3dTtvpazVzv35NuZmn|000f3dTtvpazVzv35NuZmn|
|006AgHXrEw13oyg0D8evRa|006AgHXrEw13oyg0D8evRa|
|006XcK33PV5wCV9g629nYA|006XcK33PV5wCV9g629nYA|
|006qPzTTwYGb4t4glyTaLw|006qPzTTwYGb4t4glyTaLw|
+----------------------+----------------------+

-RECORD 0-------------------
 album_id_vacio   | 1156961 
 match_album_id   | 22826   
 match_spotify_id | 22826   

+------------------------------------+-----------------------+------------+------------------+------+--------+
|track_id                            |title                  |track_artist|

Plata

In [0]:
import json
from pyspark.sql import functions as F, Window as W

CATALOG, SCHEMA = "workspace", "spotify"
s = f"{CATALOG}.{SCHEMA}"
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

quality_log = []   # se reinicia cada vez que se ejecuta esta celda

def log(paso, filas, decision):
    quality_log.append({"paso": paso, "filas_afectadas": int(filas or 0), "decision": decision})
    print(f"{paso:48s} {int(filas or 0):>10,}  → {decision}")

def null_markers(c):
    """Expresión SQL que convierte '', 'nan', 'null', 'none' en NULL real."""
    return f"CASE WHEN lower(trim(`{c}`)) IN ('', 'nan', 'null', 'none') THEN NULL ELSE `{c}` END"

In [0]:
with open(f"{RAW_PATH}/metadata/metadata.json") as fh:
    expected_types = {c["name"]: c["type"] for c in json.load(fh)["schema"]}
spark_type = {"int": "BIGINT", "float": "DOUBLE", "bool": "BOOLEAN", "string": "STRING"}

bronze = spark.table(f"{s}.bronze_tracks")
N0 = bronze.count()

data_cols = [c for c in bronze.columns if not c.startswith("_")]
typed = bronze.selectExpr(
    *[f"try_cast({null_markers(c)} AS {spark_type[expected_types.get(c, 'string')]}) AS `{c}`" for c in data_cols],
    "_ingest_ts AS bronze_ingest_ts"
)
log("Conversión de tipos (58 columnas)", 0, "Tipos de metadata.json; 0 valores inválidos (perfilado)")

Conversión de tipos (58 columnas)                         0  → Tipos de metadata.json; 0 valores inválidos (perfilado)


In [0]:
typed = typed.withColumn("title_key",
                         F.coalesce(F.col("title_normalized"), F.lower(F.trim(F.col("title")))))

w = (W.partitionBy("title_key", "track_artist")
      .orderBy(F.desc_nulls_last("track_popularity"), F.desc_nulls_last("streams_total"), "track_id"))

typed = (typed
         .withColumn("version_rank", F.row_number().over(w))
         .withColumn("is_alt_version",
                     (F.col("version_rank") > 1)
                     & F.col("title_key").isNotNull()
                     & F.col("track_artist").isNotNull()))

log("title_normalized faltante", 25_901, "title_key = coalesce(title_normalized, lower(trim(title)))")
log("Versiones alternativas (título + artista)",
    typed.filter("is_alt_version").count(),
    "Se marcan con is_alt_version; no se eliminan (decisión del análisis en oro)")

title_normalized faltante                            25,901  → title_key = coalesce(title_normalized, lower(trim(title)))
Versiones alternativas (título + artista)            19,902  → Se marcan con is_alt_version; no se eliminan (decisión del análisis en oro)


In [0]:
# Enriquecimiento: solo se unen las columnas con cobertura real (ver celda de cobertura)
feat = (spark.table(f"{s}.bronze_features")
        .selectExpr("track_id",
                    f"try_cast({null_markers('genre_fragmentation_index')} AS DOUBLE) AS genre_fragmentation_index",
                    f"{null_markers('sonic_fingerprint')} AS sonic_fingerprint"))

silver = typed.join(feat, "track_id", "left")

N1 = silver.count()
assert N1 == N0, f"Un join cambió el número de filas: {N0:,} → {N1:,}"

log("Join features (2 columnas exclusivas)", 0,
    "Join 1:1 por track_id: genre_fragmentation_index y sonic_fingerprint (100 % cobertura)")
log("artist_followers (bronze_features)", N0,
    "Descartada: 0 % de cobertura")
log("artist_id repetido en bronze_artists", 166,
    "El ID se genera desde el nombre en minúsculas y mezcla artistas distintos")
log("Join con bronze_artists", N0,
    "Descartado: país, solista y oyentes con 0 % de cobertura; se usa track_artist como clave de artista")
log("Canciones sin track_artist", 19,
    "Se conservan (audio válido); atributos de artista quedan NULL")
log("album_id faltante", 1_156_961,
    "Join con bronze_albums descartado: cubre solo 1,9 % de las canciones")
log("Nulos estructurales chart_* (3 columnas)", 1_167_441,
    "Se mantienen NULL: la canción nunca entró en un ranking")

Join features (2 columnas exclusivas)                     0  → Join 1:1 por track_id: genre_fragmentation_index y sonic_fingerprint (100 % cobertura)
artist_followers (bronze_features)                1,182,354  → Descartada: 0 % de cobertura
artist_id repetido en bronze_artists                    166  → El ID se genera desde el nombre en minúsculas y mezcla artistas distintos
Join con bronze_artists                           1,182,354  → Descartado: país, solista y oyentes con 0 % de cobertura; se usa track_artist como clave de artista
Canciones sin track_artist                               19  → Se conservan (audio válido); atributos de artista quedan NULL
album_id faltante                                 1,156,961  → Join con bronze_albums descartado: cubre solo 1,9 % de las canciones
Nulos estructurales chart_* (3 columnas)          1,167,441  → Se mantienen NULL: la canción nunca entró en un ranking


In [0]:
# Validación de rangos: se cuentan los valores fuera de rango y se registra la decisión
unit_cols = ["energy", "valence", "danceability", "acousticness",
             "instrumentalness", "speechiness", "liveness"]
checks = {f"{c} fuera de [0, 1]": (F.col(c) < 0) | (F.col(c) > 1) for c in unit_cols}
checks["tempo <= 0"]                         = F.col("tempo") <= 0
checks["loudness > 0 dB"]                    = F.col("loudness") > 0
checks["duration_ms <= 0"]                   = F.col("duration_ms") <= 0
checks["track_popularity fuera de [0, 100]"] = (F.col("track_popularity") < 0) | (F.col("track_popularity") > 100)

# Una sola pasada sobre los datos: todas las condiciones se evalúan juntas
res = silver.select([F.sum(cond.cast("int")).alias(f"c{i}")
                     for i, cond in enumerate(checks.values())]).first()

for i, name in enumerate(checks):
    n = res[f"c{i}"] or 0
    log(f"Rango: {name}", n, "Sin casos: no requiere corrección" if n == 0 else "Revisar antes de oro")

Rango: energy fuera de [0, 1]                             0  → Sin casos: no requiere corrección
Rango: valence fuera de [0, 1]                            0  → Sin casos: no requiere corrección
Rango: danceability fuera de [0, 1]                       0  → Sin casos: no requiere corrección
Rango: acousticness fuera de [0, 1]                       0  → Sin casos: no requiere corrección
Rango: instrumentalness fuera de [0, 1]                   0  → Sin casos: no requiere corrección
Rango: speechiness fuera de [0, 1]                        0  → Sin casos: no requiere corrección
Rango: liveness fuera de [0, 1]                           0  → Sin casos: no requiere corrección
Rango: tempo <= 0                                         0  → Sin casos: no requiere corrección
Rango: loudness > 0 dB                                    0  → Sin casos: no requiere corrección
Rango: duration_ms <= 0                                   0  → Sin casos: no requiere corrección
Rango: track_popularity fuera 

In [0]:
(silver.write.mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable(f"{s}.silver_tracks"))

(spark.createDataFrame(quality_log)
      .select("paso", "filas_afectadas", "decision")
      .write.mode("overwrite").option("overwriteSchema", True)
      .saveAsTable(f"{s}.silver_quality_log"))

print(f"silver_tracks: {spark.table(f'{s}.silver_tracks').count():,} filas, "
      f"{len(spark.table(f'{s}.silver_tracks').columns)} columnas")
display(spark.table(f"{s}.silver_quality_log"))

silver_tracks: 1,182,354 filas, 89 columnas


paso,filas_afectadas,decision
Conversión de tipos (58 columnas),0,Tipos de metadata.json; 0 valores inválidos (perfilado)
title_normalized faltante,25901,"title_key = coalesce(title_normalized, lower(trim(title)))"
Versiones alternativas (título + artista),19902,Se marcan con is_alt_version; no se eliminan (decisión del análisis en oro)
Join features (2 columnas exclusivas),0,Join 1:1 por track_id: genre_fragmentation_index y sonic_fingerprint (100 % cobertura)
artist_followers (bronze_features),1182354,Descartada: 0 % de cobertura
artist_id repetido en bronze_artists,166,El ID se genera desde el nombre en minúsculas y mezcla artistas distintos
Join con bronze_artists,1182354,"Descartado: país, solista y oyentes con 0 % de cobertura; se usa track_artist como clave de artista"
Canciones sin track_artist,19,Se conservan (audio válido); atributos de artista quedan NULL
album_id faltante,1156961,"Join con bronze_albums descartado: cubre solo 1,9 % de las canciones"
Nulos estructurales chart_* (3 columnas),1167441,Se mantienen NULL: la canción nunca entró en un ranking


In [0]:
# Cobertura de las columnas agregadas por el join (el resultado anterior justificó descartar las demás)
spark.sql(f"""
SELECT ROUND(100 * COUNT(genre_fragmentation_index) / COUNT(*), 2) AS pct_fragmentacion_genero,
       ROUND(100 * COUNT(sonic_fingerprint)         / COUNT(*), 2) AS pct_sonic_fingerprint
FROM {s}.silver_tracks
""").show(vertical=True)

-RECORD 0-------------------------
 pct_fragmentacion_genero | 100.0 
 pct_sonic_fingerprint    | 100.0 



%md
## Capa oro: respuestas a las preguntas del proyecto
Cada consulta se ejecuta 3 veces para medir su tiempo, se revisa su plan con `explain()`
y su resultado se guarda como tabla `gold_*` para el dashboard.

In [0]:
import time, statistics

T = f"{s}.silver_tracks"
perf_log = []   # se reinicia al ejecutar esta celda

def medir(nombre, sql, repeticiones=3):
    """Ejecuta la consulta varias veces y registra los tiempos."""
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        spark.sql(sql).collect()                     # collect() fuerza la ejecución completa
        tiempos.append(round(time.perf_counter() - t0, 3))
    perf_log.append({"consulta": nombre,
                     "t_primera_s": tiempos[0],
                     "t_mediana_resto_s": statistics.median(tiempos[1:]),
                     "tiempos_s": str(tiempos)})
    print(f"{nombre}: {tiempos} s")

def construir_gold(nombre, tabla, sql):
    """Mide la consulta, muestra su plan y guarda el resultado como tabla oro."""
    medir(nombre, sql)
    df = spark.sql(sql)
    print("\n=== Plan de ejecución ===")
    df.explain(mode="formatted")
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{s}.{tabla}")
    display(spark.table(f"{s}.{tabla}"))

In [0]:
construir_gold("P1 · Audio features por década", "gold_p1_audio_por_decada", f"""
SELECT decade,
       COUNT(*)                     AS n_canciones,
       ROUND(AVG(energy), 3)        AS energy,
       ROUND(AVG(valence), 3)       AS valence,
       ROUND(AVG(acousticness), 3)  AS acousticness,
       ROUND(AVG(danceability), 3)  AS danceability
FROM {T}
WHERE NOT is_alt_version
GROUP BY decade
ORDER BY decade
""")

P1 · Audio features por década: [1.166, 0.954, 0.923] s

=== Plan de ejecución ===
== Physical Plan ==
AdaptiveSparkPlan (14)
+- == Initial Plan ==
   PhotonResultStage (13)
   +- PhotonColumnarToRow (12)
      +- PhotonSort (11)
         +- PhotonShuffleExchangeSource (10)
            +- PhotonShuffleMapStage (9)
               +- PhotonShuffleExchangeSink (8)
                  +- PhotonGroupingAgg (7)
                     +- PhotonShuffleExchangeSource (6)
                        +- PhotonShuffleMapStage (5)
                           +- PhotonShuffleExchangeSink (4)
                              +- PhotonGroupingAgg (3)
                                 +- PhotonProject (2)
                                    +- PhotonScan parquet workspace.spotify.silver_tracks (1)


(1) PhotonScan parquet workspace.spotify.silver_tracks
Output [6]: [energy#37250, valence#37251, danceability#37252, acousticness#37253, decade#37278L, is_alt_version#37323]
DictionaryFilters: [NOT is_alt_version#37323]

decade,n_canciones,energy,valence,acousticness,danceability
1950,2,0.772,0.911,0.424,0.606
1960,110,0.598,0.614,0.337,0.503
1970,640,0.625,0.604,0.252,0.541
1980,924,0.689,0.632,0.161,0.616
1990,1734,0.662,0.598,0.145,0.672
2000,439240,0.631,0.487,0.333,0.531
2010,520654,0.652,0.44,0.306,0.542
2020,199148,0.633,0.426,0.323,0.549


In [0]:
medir("P2 · optimizada (sin ORDER BY)", f"""
SELECT genre_l1,
       COUNT(*)                         AS n_canciones,
       ROUND(AVG(energy), 3)            AS energy,
       ROUND(AVG(valence), 3)           AS valence,
       ROUND(AVG(danceability), 3)      AS danceability,
       ROUND(AVG(acousticness), 3)      AS acousticness,
       ROUND(AVG(instrumentalness), 3)  AS instrumentalness,
       ROUND(AVG(speechiness), 3)       AS speechiness,
       ROUND(AVG(liveness), 3)          AS liveness,
       ROUND(AVG(tempo), 1)             AS tempo_bpm,
       ROUND(AVG(loudness), 1)          AS loudness_db
FROM {T}
WHERE NOT is_alt_version AND genre_l1 IS NOT NULL
GROUP BY genre_l1
""")
spark.sql(f"SELECT genre_l1, COUNT(*) FROM {T} WHERE NOT is_alt_version GROUP BY genre_l1").explain(mode="formatted")

P2 · optimizada (sin ORDER BY): [1.133, 0.835, 0.884] s
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonScan parquet workspace.spotify.silver_tracks (1)


(1) PhotonScan parquet workspace.spotify.silver_tracks
Output [2]: [genre_l1#40090, is_alt_version#40149]
DictionaryFilters: [NOT is_alt_version#40149]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-q6beoakyin/uc/a8080afe-0d40-479d-a646-7835cc019665/8c4561f9-0d6d-4e2f-9a75-2de19f821988/__unitystorage/catalogs/2d48c4db-b469-4461-8d22-d6ba41136437/tables/a6f4e870-d19c-422e-bd87-f8127da3a492]
ReadSchema: struct<genre_l1:string,is_alt_version:boolean>
RequiredDataFilters: [isnotnull(

In [0]:
construir_gold("P2 · Perfil sonoro por género L1", "gold_p2_perfil_por_genero", f"""
SELECT genre_l1,
       COUNT(*)                         AS n_canciones,
       ROUND(AVG(energy), 3)            AS energy,
       ROUND(AVG(valence), 3)           AS valence,
       ROUND(AVG(danceability), 3)      AS danceability,
       ROUND(AVG(acousticness), 3)      AS acousticness,
       ROUND(AVG(instrumentalness), 3)  AS instrumentalness,
       ROUND(AVG(speechiness), 3)       AS speechiness,
       ROUND(AVG(liveness), 3)          AS liveness,
       ROUND(AVG(tempo), 1)             AS tempo_bpm,
       ROUND(AVG(loudness), 1)          AS loudness_db
FROM {T}
WHERE NOT is_alt_version AND genre_l1 IS NOT NULL
GROUP BY genre_l1
ORDER BY n_canciones DESC
""")

P2 · Perfil sonoro por género L1: [0.952, 0.888, 0.844] s

=== Plan de ejecución ===
== Physical Plan ==
AdaptiveSparkPlan (14)
+- == Initial Plan ==
   PhotonResultStage (13)
   +- PhotonColumnarToRow (12)
      +- PhotonSort (11)
         +- PhotonShuffleExchangeSource (10)
            +- PhotonShuffleMapStage (9)
               +- PhotonShuffleExchangeSink (8)
                  +- PhotonGroupingAgg (7)
                     +- PhotonShuffleExchangeSource (6)
                        +- PhotonShuffleMapStage (5)
                           +- PhotonShuffleExchangeSink (4)
                              +- PhotonGroupingAgg (3)
                                 +- PhotonProject (2)
                                    +- PhotonScan parquet workspace.spotify.silver_tracks (1)


(1) PhotonScan parquet workspace.spotify.silver_tracks
Output [11]: [energy#42272, valence#42273, danceability#42274, acousticness#42275, instrumentalness#42276, speechiness#42277, liveness#42278, loudness#42279, temp

genre_l1,n_canciones,energy,valence,danceability,acousticness,instrumentalness,speechiness,liveness,tempo_bpm,loudness_db
Electronic,303901,0.677,0.401,0.586,0.206,0.457,0.077,0.191,125.3,-9.1
Rock,133079,0.737,0.491,0.484,0.178,0.113,0.069,0.237,126.9,-7.0
Pop,127936,0.578,0.465,0.523,0.453,0.129,0.169,0.279,116.1,-10.6
Metal,97443,0.885,0.272,0.357,0.032,0.318,0.115,0.256,124.8,-5.7
Latin,90450,0.644,0.649,0.603,0.478,0.069,0.076,0.283,122.2,-7.6
World,78963,0.603,0.489,0.56,0.372,0.236,0.096,0.2,119.3,-9.5
Classical,57492,0.271,0.28,0.383,0.795,0.404,0.067,0.196,109.2,-17.0
Folk,50749,0.447,0.432,0.532,0.555,0.113,0.05,0.188,118.5,-10.3
R&B,39881,0.675,0.536,0.624,0.224,0.175,0.101,0.202,118.7,-7.5
Reggae,37263,0.692,0.571,0.678,0.163,0.167,0.163,0.197,121.0,-7.0


In [0]:
construir_gold("P3 · Audio por banda de popularidad", "gold_p3_audio_por_popularidad", f"""
SELECT CASE WHEN track_popularity < 10 THEN '1. 0-9'
            WHEN track_popularity < 30 THEN '2. 10-29'
            WHEN track_popularity < 50 THEN '3. 30-49'
            WHEN track_popularity < 70 THEN '4. 50-69'
            ELSE '5. 70-100' END        AS banda_popularidad,
       COUNT(*)                         AS n_canciones,
       ROUND(AVG(energy), 3)            AS energy,
       ROUND(AVG(valence), 3)           AS valence,
       ROUND(AVG(danceability), 3)      AS danceability,
       ROUND(AVG(acousticness), 3)      AS acousticness,
       ROUND(AVG(instrumentalness), 3)  AS instrumentalness,
       ROUND(AVG(speechiness), 3)       AS speechiness
FROM {T}
WHERE NOT is_alt_version AND track_popularity IS NOT NULL
GROUP BY 1
ORDER BY 1
""")

P3 · Audio por banda de popularidad: [1.106, 0.914, 0.863] s

=== Plan de ejecución ===
== Physical Plan ==
AdaptiveSparkPlan (14)
+- == Initial Plan ==
   PhotonResultStage (13)
   +- PhotonColumnarToRow (12)
      +- PhotonSort (11)
         +- PhotonShuffleExchangeSource (10)
            +- PhotonShuffleMapStage (9)
               +- PhotonShuffleExchangeSink (8)
                  +- PhotonGroupingAgg (7)
                     +- PhotonShuffleExchangeSource (6)
                        +- PhotonShuffleMapStage (5)
                           +- PhotonShuffleExchangeSink (4)
                              +- PhotonGroupingAgg (3)
                                 +- PhotonProject (2)
                                    +- PhotonScan parquet workspace.spotify.silver_tracks (1)


(1) PhotonScan parquet workspace.spotify.silver_tracks
Output [8]: [energy#45243, valence#45244, danceability#45245, acousticness#45246, instrumentalness#45247, speechiness#45248, track_popularity#45260L, is_alt_ve

banda_popularidad,n_canciones,energy,valence,danceability,acousticness,instrumentalness,speechiness
1. 0-9,419183,0.635,0.472,0.534,0.343,0.3,0.098
2. 10-29,455792,0.646,0.44,0.521,0.312,0.259,0.092
3. 30-49,228999,0.641,0.451,0.565,0.297,0.19,0.085
4. 50-69,53791,0.641,0.486,0.62,0.276,0.086,0.099
5. 70-100,4687,0.65,0.518,0.653,0.244,0.022,0.101


In [0]:
construir_gold("P4 · Emotional arc por era", "gold_p4_emocion_por_era", f"""
SELECT era,
       emotional_arc,
       COUNT(*) AS n_canciones,
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY era), 2) AS pct_en_era
FROM {T}
WHERE NOT is_alt_version AND emotional_arc IS NOT NULL
GROUP BY era, emotional_arc
ORDER BY era, emotional_arc
""")

P4 · Emotional arc por era: [1.142, 0.834, 1.03] s

=== Plan de ejecución ===
== Physical Plan ==
AdaptiveSparkPlan (19)
+- == Initial Plan ==
   PhotonResultStage (18)
   +- PhotonColumnarToRow (17)
      +- PhotonSort (16)
         +- PhotonShuffleExchangeSource (15)
            +- PhotonShuffleMapStage (14)
               +- PhotonShuffleExchangeSink (13)
                  +- PhotonWindow (12)
                     +- PhotonSort (11)
                        +- PhotonShuffleExchangeSource (10)
                           +- PhotonShuffleMapStage (9)
                              +- PhotonShuffleExchangeSink (8)
                                 +- PhotonGroupingAgg (7)
                                    +- PhotonShuffleExchangeSource (6)
                                       +- PhotonShuffleMapStage (5)
                                          +- PhotonShuffleExchangeSink (4)
                                             +- PhotonGroupingAgg (3)
                                       

era,emotional_arc,n_canciones,pct_en_era
2000-2010,Q1_euphoric,190739,39.27
2000-2010,Q2_angry,147396,30.34
2000-2010,Q3_depressed,105247,21.67
2000-2010,Q4_content,42373,8.72
2011-2020,Q1_euphoric,178526,33.74
2011-2020,Q2_angry,205568,38.85
2011-2020,Q3_depressed,110937,20.96
2011-2020,Q4_content,34161,6.46
2021+,Q1_euphoric,46675,32.39
2021+,Q2_angry,55390,38.44


In [0]:
construir_gold("P5 · Top artistas por hit_rate", "gold_p5_top_artistas", f"""
SELECT track_artist,
       genre_l1_mas_frecuente,
       n_canciones,
       hit_rate,
       popularidad_media
FROM (
  SELECT track_artist,
         MODE(genre_l1)                   AS genre_l1_mas_frecuente,
         COUNT(*)                         AS n_canciones,
         ROUND(MAX(hit_rate), 3)          AS hit_rate,
         ROUND(AVG(track_popularity), 1)  AS popularidad_media
  FROM {T}
  WHERE NOT is_alt_version AND track_artist IS NOT NULL
  GROUP BY track_artist
  HAVING COUNT(*) >= 20
)
ORDER BY hit_rate DESC, popularidad_media DESC
LIMIT 20
""")

P5 · Top artistas por hit_rate: [1.308, 1.011, 0.953] s

=== Plan de ejecución ===
== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonTopK (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonTopK (9)
                     +- PhotonFilter (8)
                        +- PhotonGroupingAgg (7)
                           +- PhotonShuffleExchangeSource (6)
                              +- PhotonShuffleMapStage (5)
                                 +- PhotonShuffleExchangeSink (4)
                                    +- PhotonGroupingAgg (3)
                                       +- PhotonProject (2)
                                          +- PhotonScan parquet workspace.spotify.silver_tracks (1)


(1) PhotonScan parquet workspace.spotify.silver_tracks
Output [5]: [track_artist#50116, genre_l1#5

track_artist,genre_l1_mas_frecuente,n_canciones,hit_rate,popularidad_media
Harry Styles,Pop,35,0.842,76.0
Feid,Pop,24,0.792,71.5
SZA,Pop,53,0.604,70.4
Bad Bunny,Latin,30,0.576,59.1
Billie Eilish,Electronic,61,0.529,69.0
Bizarrap,Hip-Hop,23,0.522,68.7
Rauw Alejandro,Pop,35,0.514,67.1
Morgan Wallen,Country,90,0.473,67.9
Rex Orange County,Pop,21,0.444,67.8
J Balvin,Latin,33,0.38,57.1


In [0]:
# 1. Misma tabla plata, pero particionada por genre_l1
(spark.table(T).write.mode("overwrite").option("overwriteSchema", True)
      .partitionBy("genre_l1")
      .saveAsTable(f"{s}.silver_tracks_by_genre"))

# 2. Tamaño y número de archivos de cada versión
for tabla in [T, f"{s}.silver_tracks_by_genre"]:
    d = spark.sql(f"DESCRIBE DETAIL {tabla}").select("name", "numFiles", "sizeInBytes", "partitionColumns").first()
    print(f"{d.name}: {d.numFiles} archivos, {d.sizeInBytes/1024/1024:.1f} MB, particiones={d.partitionColumns}")

# 3. La misma consulta con filtro por género, sobre las dos tablas
q = "SELECT COUNT(*) AS n, AVG(energy) AS energy, AVG(valence) AS valence FROM {tabla} WHERE genre_l1 = 'Rock'"
medir("Exp · Filtro Rock · SIN particionar", q.format(tabla=T))
medir("Exp · Filtro Rock · particionada por genre_l1", q.format(tabla=f"{s}.silver_tracks_by_genre"))

print("\n=== Plan: tabla particionada ===")
spark.sql(q.format(tabla=f"{s}.silver_tracks_by_genre")).explain(mode="formatted")

workspace.spotify.silver_tracks: 4 archivos, 249.6 MB, particiones=[]
workspace.spotify.silver_tracks_by_genre: 18 archivos, 250.2 MB, particiones=['genre_l1']
Exp · Filtro Rock · SIN particionar: [0.732, 0.737, 0.645] s
Exp · Filtro Rock · particionada por genre_l1: [0.825, 0.648, 0.626] s

=== Plan: tabla particionada ===
== Physical Plan ==
AdaptiveSparkPlan (6)
+- == Initial Plan ==
   PhotonResultStage (5)
   +- PhotonColumnarToRow (4)
      +- PhotonAgg (3)
         +- PhotonProject (2)
            +- PhotonScan parquet workspace.spotify.silver_tracks_by_genre (1)


(1) PhotonScan parquet workspace.spotify.silver_tracks_by_genre
Output [3]: [energy#56829, valence#56830, genre_l1#56843]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-q6beoakyin/uc/a8080afe-0d40-479d-a646-7835cc019665/8c4561f9-0d6d-4e2f-9a75-2de19f821988/__unitystorage/catalogs/2d48c4db-b469-4461-8d22-d6ba41136437/tables/a2a22346-f51b-49ef-879e-a842f8dc35f1]
PartitionFilters: [isnotnull(genre_l1#56843), (genr

In [0]:
(spark.createDataFrame(perf_log)
      .select("consulta", "t_primera_s", "t_mediana_resto_s", "tiempos_s")
      .write.mode("overwrite").option("overwriteSchema", True)
      .saveAsTable(f"{s}.gold_performance"))
display(spark.table(f"{s}.gold_performance"))

consulta,t_primera_s,t_mediana_resto_s,tiempos_s
P1 · Audio features por década,1.166,0.9385,"[1.166, 0.954, 0.923]"
P2 · optimizada (sin ORDER BY),1.133,0.8594999999999999,"[1.133, 0.835, 0.884]"
P2 · Perfil sonoro por género L1,0.952,0.866,"[0.952, 0.888, 0.844]"
P3 · Audio por banda de popularidad,1.106,0.8885000000000001,"[1.106, 0.914, 0.863]"
P4 · Emotional arc por era,1.142,0.9319999999999999,"[1.142, 0.834, 1.03]"
P5 · Top artistas por hit_rate,1.308,0.982,"[1.308, 1.011, 0.953]"
Exp · Filtro Rock · SIN particionar,0.732,0.6910000000000001,"[0.732, 0.737, 0.645]"
Exp · Filtro Rock · particionada por genre_l1,0.825,0.637,"[0.825, 0.648, 0.626]"


In [0]:
medir("P2 · optimizada (sin ORDER BY)", f"""
SELECT genre_l1,
       COUNT(*)                         AS n_canciones,
       ROUND(AVG(energy), 3)            AS energy,
       ROUND(AVG(valence), 3)           AS valence,
       ROUND(AVG(danceability), 3)      AS danceability,
       ROUND(AVG(acousticness), 3)      AS acousticness,
       ROUND(AVG(instrumentalness), 3)  AS instrumentalness,
       ROUND(AVG(speechiness), 3)       AS speechiness,
       ROUND(AVG(liveness), 3)          AS liveness,
       ROUND(AVG(tempo), 1)             AS tempo_bpm,
       ROUND(AVG(loudness), 1)          AS loudness_db
FROM {T}
WHERE NOT is_alt_version AND genre_l1 IS NOT NULL
GROUP BY genre_l1
""")
spark.sql(f"SELECT genre_l1, COUNT(*) FROM {T} WHERE NOT is_alt_version GROUP BY genre_l1").explain(mode="formatted")

P2 · optimizada (sin ORDER BY): [0.867, 0.849, 1.089] s
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonScan parquet workspace.spotify.silver_tracks (1)


(1) PhotonScan parquet workspace.spotify.silver_tracks
Output [2]: [genre_l1#59268, is_alt_version#59327]
DictionaryFilters: [NOT is_alt_version#59327]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-q6beoakyin/uc/a8080afe-0d40-479d-a646-7835cc019665/8c4561f9-0d6d-4e2f-9a75-2de19f821988/__unitystorage/catalogs/2d48c4db-b469-4461-8d22-d6ba41136437/tables/a6f4e870-d19c-422e-bd87-f8127da3a492]
ReadSchema: struct<genre_l1:string,is_alt_version:boolean>
RequiredDataFilters: [isnotnull(